## StructureDCA: Ready-to-Use Notebook
[![PyPi Version](https://img.shields.io/pypi/v/structuredca.svg)](https://pypi.org/project/structuredca/) [![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/3BioCompBio/StructureDCA/blob/main/colab_notebook_StructureDCA.ipynb)

<img src="https://raw.githubusercontent.com/3BioCompBio/StructureDCA/main/Logo.png" height="350" align="right" style="height:200px;">

Ready-to-Use Notebook to run the StructureDCA model:
 - Upload or fetch an MSA file
 - Upload or fetch a 3D structure file
 - Run predictions on all single-site mutations
 - Visualize results

The `structuredca` Python package implements **Structure-Informed Direct Coupling Analysis** (StructureDCA) to predict the **effects of missense mutations on proteins**.

Standard DCA methods use **Multiple Sequence Alignments (MSAs)** to build a **statistical evolutionary model** of homologous protein families. They rely on single-site fields `h` and pairwise couplings `J` that capture co-evolution between residue positions.
StructureDCA extends this framework by incorporating the **residue–residue contact map** derived from the protein **3D structure** to infer a **sparse DCA model**, in which couplings between spatially distant residue pairs are removed.
This approach leverages the observation that functionally relevant, co-evolving residues are most often structurally in contact.

The package includes a **pseudolikelihood-maximization DCA solver** capable of inferring sparse DCA models, where selected coupling coefficients `Jij` are constrained to zero.
StructureDCA combines a flexible, user-friendly Python interface with the high computational efficiency of its C++ backend.
This model was initially developed to improve classical DCA methods for predicting the effects of missense mutations in proteins. However, StructureDCA can be applied to any DCA-based analysis (except for contact predictions...) and provides the full functionality of both standard and sparse DCA models.

**Please cite**:
- [Matsvei Tsishyn, Hugo Talibart, Marianne Rooman, Fabrizio Pucci. Inferring Protein Mutational Landscape with Structure-Informed Direct Coupling Analysis. BioRxiv](https://www.biorxiv.org/).

In [ ]:
#@title Install and Import dependencies

# Install ----------------------------------------------------------------------
%pip install structuredca


# Imports ----------------------------------------------------------------------
import os
import tarfile
import time
from typing import Callable
import requests
from requests import Response
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output
from Bio.PDB import PDBParser, PDBList
from Bio.PDB.Structure import Structure
from structuredca import StructureDCA
from structuredca.sequence import Sequence, FastaReader, FastaStream, PairwiseAlignment, AminoAcid
from structuredca.sequence.mutation import read_mutations_file



# Init paths -------------------------------------------------------------------
# Set these paths to None to prevent the user from executing cells in incorrect order
msa_path = None
pdb_path = None
chain = None
output_path = None


# Dependencies (small helper functions) ----------------------------------------
def clip_string(input_str: str, max_len: int=100) -> str:
  """Truncate a string and append '...' if it exceeds max_len."""
  return input_str if len(input_str) <= max_len else input_str[:max_len] + "..."

def extract_sequences(structure: Structure, pdb_name: str="pdb") -> dict[str, Sequence]:
  """Extract the standardized amino acid sequence for each chain in the PDB Structure (uses only first model)."""
  sequences_by_chain: dict[str, Sequence] = {}
  for chain in structure[0]: # loop only on model 1
    aa_seq: list[AminoAcid] = []
    for residue in chain:
      aa = AminoAcid.parse_three(residue.resname)
      if aa.is_aminoacid() and not aa.is_unknown():
        aa_seq.append(aa)
    sequences_by_chain[str(chain.id)] = Sequence(f"{pdb_name}_{chain.id}", "".join(aa.one for aa in aa_seq))
  return sequences_by_chain

# Dependencies (fetch functions) -----------------------------------------------
def is_valid_pdb_id(pdb_id :str) -> bool:
  """Returns if 'input' can be a PDB-ID."""
  POSSIBLE_FIRST_CHARACTERS = "123456789"
  POSSIBLE_FOLLOWING_CHARACTERS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
  input_upper = pdb_id.upper()
  return len(input_upper) == 4 \
    and input_upper[0] in POSSIBLE_FIRST_CHARACTERS \
    and all([char in POSSIBLE_FOLLOWING_CHARACTERS for char in input_upper[1:]])

def fetch_structure_by_pdb(pdb_id: str) -> str:
  """Fetch a '.pdb' file from the PDB and return its path."""
  pdb_id = pdb_id.lower().strip()
  if not is_valid_pdb_id(pdb_id):
    raise ValueError(f"❌ pdb_id='{pdb_id}' is not a valid PDB Id.")
  pdb_fetcher = PDBList()
  file_path = pdb_fetcher.retrieve_pdb_file(pdb_id, file_format="pdb", pdir="./")
  if file_path is None:
    raise ValueError(f"❌ Fetch pdb_id='{pdb_id}' has failed.")
  if file_path.endswith(".ent"):
    file_path_old = file_path
    file_path = file_path.removesuffix(".ent") + ".pdb"
    os.rename(file_path_old, file_path)
  return file_path

def fetch_structure_by_uniprot(uniprot_id: str) -> str:
  """Fetch a '.pdb' file from the AlphaFoldDB by its UniProt ID and return its path."""

  # Init
  uniprot_id = uniprot_id.upper().strip()
  filename = f"AF-{uniprot_id}-F1-model_v6.pdb"
  url = f"https://alphafold.ebi.ac.uk/files/{filename}"
  print(f" * fetch 3D structure from '{url}'")

  # Request structure
  try:
    r = requests.get(url)
    r.raise_for_status()
  except Exception as err:
    raise ValueError(f"❌ AlphaFoldDB structure fetch failed for UniProt ID '{uniprot_id}' from '{url}': {err}")

  # Save structure and return path
  with open(filename, "wb") as fs:
    fs.write(r.content)
  return filename

def fetch_msa_by_uniprot(uniprot_id: str) -> str:
  """Fetch an MSA file from the AlphaFoldDB by its UniProt ID and return its path."""

  # Init
  uniprot_id = uniprot_id.upper().strip()
  filename = f"AF-{uniprot_id}-F1-msa_v6.a3m"
  url = f"https://alphafold.ebi.ac.uk/files/msa/{filename}"
  print(f" * fetch MSA from '{url}'")

  # Steam and write MSA to file and return path
  try:
    with requests.get(url, stream=True) as resp:
      resp.raise_for_status()
      with open(filename, "wb") as fs:
        for chunk in resp.iter_content(1 << 14): # 16 KB chunks
          if chunk:
            fs.write(chunk)
  except Exception as err:
    raise ValueError(f"❌ AlphaFoldDB MSA fetch failed for UniProt ID '{uniprot_id}': download failed from '{url}': {err}")
  return filename

def fetch_sequence_by_uniprot(uniprot_id: str) -> str:
  """Fetch an FASTA sequence from UniProt by its UniProt ID and return its path."""

  # Init
  uniprot_id = uniprot_id.upper().strip()
  filename = f"{uniprot_id}.fasta"
  url = f"https://rest.uniprot.org/uniprotkb/{filename}"
  print(f" * fetch FASTA sequence from '{url}'")

  # Request fasta
  try:
    r = requests.get(url)
    r.raise_for_status()
  except Exception as err:
    raise ValueError(f"❌ UniProt FASTA sequence fetch failed for UniProt ID '{uniprot_id}' from '{url}': {err}")

  # Save fasta and return path
  with open(filename, "wb") as fs:
    fs.write(r.content)
  return filename

# Dependencies (MMSeqs2 API) ---------------------------------------------------

def repeated_request(request_function: Callable):
  """Decorator to repeat a request 5 times before trowing an error."""

  def wrapper(*args, **kwargs) -> Response:
    MAX_REQUEST_TRIES = 5
    FAIL_SLEEP_TIME = 5.0
    print(f" * Request [{request_function.__name__}] for '{clip_string(args[0])}' ({MAX_REQUEST_TRIES} repeats)")
    for i in range(MAX_REQUEST_TRIES):
      print(f"   - [{request_function.__name__}] repeat [{i+1}/{MAX_REQUEST_TRIES}] ...")
      try:
        res: Response = request_function(*args, **kwargs)
      except Exception as err:
        print(f"   - [{request_function.__name__}] Error on attempt [{i+1}/{MAX_REQUEST_TRIES}]: {err}")
        time.sleep(FAIL_SLEEP_TIME)
      else:
        print(f"   - [{request_function.__name__}] status: {res.status_code}")
        return res
    raise ValueError(f"Too many failed attempts ({MAX_REQUEST_TRIES}) for request '{request_function.__name__}'.")

  return wrapper

def json_request(request_function: Callable):
  """Decorator run a request and parse output as JSON if possible."""

  def wrapper(*args, **kwargs):
    res: Response = request_function(*args, **kwargs)
    try:
      return res.json()
    except Exception as err:
      msg = (
        f"Failed to parse response from '{request_function.__name__}' as JSON.\n"
        f"Original error: {err}\n"
        f"HTTP status: {res.status_code}\n"
        f"Response (truncated): {clip_string(res.text)}"
      )
      raise ValueError(msg) from err

  return wrapper

@json_request
@repeated_request
def submit_mmseqs2(seq: str, mode: str, url: str, endpoint: str, timeout: float, query_name: str="query"):
  query_name = query_name.strip().replace(" ", "_").replace("\t", "_").replace("\n", "_")
  assert len(query_name) > 0, f"ERROR in submit_mmseqs2(): invalid query_name='{query_name}'"
  return requests.post(
    f"{url}/{endpoint}",
    data={"q": f">{query_name}\n{seq}\n", "mode": mode},
    timeout=timeout,
  )

@json_request
@repeated_request
def get_mmseqs2_status(id: str, url: str, timeout: float):
  res = requests.get(
    f"{url}/ticket/{id}",
    timeout=timeout,
  )
  return res

@repeated_request
def download_from_mmseqs2(id: str, url: str, timeout: float) -> Response:
  res = requests.get(
    f"{url}/result/download/{id}",
    timeout=timeout,
  )
  return res

class MMSeqs2API:

  # Constructor
  def __init__(
    self,
    tmp_dir: str,
    n_status_loop: int=100,
    loop_timeout: float = 5.0,
    single_request_timeout: float=6.02,
    use_env: bool = False,
    use_filter: bool = True,
    url: str="https://api.colabfold.com",
    endpoint: str="ticket/msa",
  ):

    # Set properties
    self.tmp_dir = tmp_dir
    self.url = url
    self.endpoint = endpoint
    self.n_status_loop = n_status_loop
    self.loop_timeout = loop_timeout
    self.single_request_timeout = single_request_timeout
    self.use_env = use_env
    self.use_filter = use_filter
    if self.use_filter:
      self.mode = "env" if self.use_env else "all"
    else:
      self.mode = "env-nofilter" if self.use_env else "nofilter"

    # Init tmp directory
    if not os.path.isdir(self.tmp_dir):
      os.mkdir(self.tmp_dir)

    # Init paths
    self.tar_gz_file = os.path.join(self.tmp_dir, "out.tar.gz")
    if not self.use_env:
      self.a3m_files = [os.path.join(self.tmp_dir, "uniref.a3m")]
    else:
      self.a3m_files = [
        os.path.join(self.tmp_dir, "uniref.a3m"),
        os.path.join(self.tmp_dir, "bfd.mgnify30.metaeuk30.smag30.a3m")
      ]

  # Run API
  def run(
    self,
    sequence: str,
    save_path: str,
    query_name: str="query",
  ) -> str:

    # Log
    print(f"Run MSA API to MMSeqs2 server '{self.url}/{self.endpoint}': ")
    print(f"   - sequence: '{clip_string(sequence)}'")
    print(f"   - save_path: '{save_path}'")

    # Submit to MMSeqs2 API
    print("Submit request:")
    out = submit_mmseqs2(sequence, self.mode, self.url, self.endpoint, self.single_request_timeout, query_name=query_name)
    status, id = out["status"], out["id"]
    if status in ["ERROR", "MAINTENANCE"]:
      raise Exception(f"MMseqs2 API is giving errors with API status: '{status}' (please try again later)")

    # Wait for request to be completed
    print("Wait for request to be completed:")
    for i in range(self.n_status_loop):
      if status not in ["UNKNOWN", "RUNNING", "PENDING"]:
        break
      print(f"   - sleep [{i+1}/{self.n_status_loop}] for {self.loop_timeout:.1f} sec. (status='{status}') ...")
      time.sleep(self.loop_timeout)
      out = get_mmseqs2_status(id, self.url, self.single_request_timeout)
      status = out["status"]

    # Status loop finished and request still not complete
    if status in ["UNKNOWN", "RUNNING", "PENDING"]:
      msg = (
        f"MMseqs2 API ERROR: maximum number of sleep loop exceeded.\n"
        f" - {self.n_status_loop} loops (of {self.loop_timeout:.1f} sec. each)\n"
        f" - request ID='{id}'\n"
      )
      raise Exception(msg)

    # Final API status check
    if status != "COMPLETE":
      raise Exception(f"MMseqs2 API is giving errors with API status: '{status}' (please try again later)")

    # Download results
    print("Download MSA:")
    download_output = download_from_mmseqs2(id, self.url, self.single_request_timeout)
    with open(self.tar_gz_file, "wb") as fs:
      fs.write(download_output.content)

    # Extract a3m files from '.tar.gz'
    print(f"Collect MSA data:")
    with tarfile.open(self.tar_gz_file) as tar_gz:
      tar_gz.extractall(self.tmp_dir, filter="data")

    # Collect and return all '.a3m' lines
    a3m_lines: list[str] = []
    for a3m_file in self.a3m_files:
      for line in open(a3m_file, "r"):
        if len(line) == 0: continue
        line = line.replace("\x00", "")
        a3m_lines.append(line)
    msa_str = "".join(a3m_lines)

    # Save MSA to file
    print(f"Save MSA to file:")
    print(f"   - save_path: '{save_path}'")
    with open(save_path, "w") as fs:
      fs.write(msa_str)

    # Return
    return save_path


In [ ]:
#@title ▶️ Run this cell to load the 3D structure

load_pdb_method_dropdown = widgets.Dropdown(
    options=[
        ("Upload local file", "upload_pdb_local_file"),
        ("Fetch from PDB", "fetch_from_pdb"),
        ("Fetch from AlphaFold DB", "fetch_pdb_from_alphafold_db"),
    ],
    value="upload_pdb_local_file",
    description="Method:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

pdb_file_upload = widgets.FileUpload(
    accept=".pdb",
    multiple=False,
    description="Upload PDB file...",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

pdb_id_input = widgets.Text(
    placeholder="PDB ID like '2cdt'",
    description="PDB ID:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

pdb_uniprot_id_input = widgets.Text(
    placeholder="UniProt ID like 'Q9LW00'",
    description="UniProt ID:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

pdb_run_button = widgets.Button(
    description="Load structure",
    button_style="primary",
    icon="check",
    style=widgets.ButtonStyle(button_color="#1a73e8")
)

pdb_cell_output = widgets.Output()

pdb_upload_section = widgets.VBox([pdb_file_upload])
pdb_section = widgets.VBox([pdb_id_input])
pdb_uniprot_section = widgets.VBox([pdb_uniprot_id_input])

def on_pdb_method_change(change):
    method = change["new"]
    pdb_upload_section.layout.display = "flex" if method == "upload_pdb_local_file" else "none"
    pdb_section.layout.display = "flex" if method == "fetch_from_pdb" else "none"
    pdb_uniprot_section.layout.display = "flex" if method == "fetch_pdb_from_alphafold_db" else "none"

load_pdb_method_dropdown.observe(on_pdb_method_change, names="value")
on_pdb_method_change({"new": load_pdb_method_dropdown.value})

def on_pdb_run_button_clicked(b):
    with pdb_cell_output:
        clear_output()
        upload_method = load_pdb_method_dropdown.value
        retrieval_id = pdb_id_input.value if upload_method == "fetch_from_pdb" else pdb_uniprot_id_input.value

        global pdb_path, pdb_name, structure, sequences_by_chain, chain

        if upload_method == "fetch_pdb_from_alphafold_db":
          global uniprot_id
          uniprot_id = pdb_uniprot_id_input.value

        pdb_path = None
        if upload_method == "upload_pdb_local_file":
            if not pdb_file_upload.value:
                print("❌ Please upload a file first.")
                return
            uploaded_file = list(pdb_file_upload.value.values())[0]
            pdb_path = uploaded_file["metadata"]["name"]
            if not pdb_path.endswith(".pdb"):
                print(f"❌ Uploaded file '{pdb_path}' should have extension '.pdb'.")
                return
            with open(pdb_path, "wb") as f:
                f.write(uploaded_file["content"])
            pdb_name = pdb_path.removesuffix(".pdb")
            print(f" * ✅ PDB file uploaded: '{pdb_path}'")

        elif upload_method == "fetch_from_pdb":
            if not retrieval_id:
                print("❌ Please specify a PDB ID.")
                return
            pdb_path = fetch_structure_by_pdb(retrieval_id)
            pdb_name = pdb_path.removesuffix(".pdb")
            print(f" * ✅ PDB file fetched: '{pdb_path}'")

        elif upload_method == "fetch_pdb_from_alphafold_db":
            if not retrieval_id:
                print("❌ Please specify a UniProt ID.")
                return
            pdb_path = fetch_structure_by_uniprot(retrieval_id)
            pdb_name = pdb_path.removesuffix(".pdb")
            print(f" * ✅ AF-DB file fetched: '{pdb_path}'")

        structure = PDBParser(QUIET=True).get_structure("protein", pdb_path)
        sequences_by_chain = extract_sequences(structure, pdb_name)
        print(f" * ✅ Choose target chain among {len(sequences_by_chain)} detected chain(s): '{''.join(sequences_by_chain.keys())}'")
        for c, seq in sequences_by_chain.items():
            print(f"    - chain {c} (L={len(seq)}): '{seq.sequence}'")

        if len(sequences_by_chain) == 1:
            chain = list(sequences_by_chain.keys())[0]
            print(f" * ✅ Target chain set to: chain='{chain}'")

pdb_run_button.on_click(on_pdb_run_button_clicked)

display(widgets.VBox([
    widgets.HTML("<h2>3D structure options</h2>"),
    load_pdb_method_dropdown,
    pdb_upload_section,
    pdb_section,
    pdb_uniprot_section,
    pdb_run_button,
    pdb_cell_output,
]))

In [ ]:
#@title ▶️ Run this cell to load the MSA

load_msa_method_dropdown = widgets.Dropdown(
    options=[
        ("Upload local file", "upload_msa_local_file"),
        ("Fetch from AlphaFold DB", "fetch_msa_from_alphafold_db"),
        ("Run MMseqs2 API", "run_mmseqs2_api")
        ],
    value="upload_msa_local_file",
    description="Method:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

msa_file_upload = widgets.FileUpload(
    accept=",".join(FastaStream.ACCEPTED_MSA_EXTENSIONS),
    multiple=False,
    description="Upload MSA file...",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

# pre-fill with UniProt ID if already filled in MSA form
if "uniprot_id" in globals():
  prefill_uid = uniprot_id
else:
  prefill_uid = ""

msa_uniprot_id_input = widgets.Text(
    placeholder="UniProt ID like 'Q9LW00'",
    description="UniProt ID:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
    value=prefill_uid
)


# pre-fill with sequence from PDB file if uploaded
try:
  for c, seq in sequences_by_chain.items():
    prefill_seq = seq.sequence
    prefill_chain = c
    break
except Exception as e:
  prefill_seq = ""
  prefill_chain = None

if prefill_chain:
  sequence_warning = widgets.HTML(f"<i>(pre-filled with chain {prefill_chain} from PDB structure)</i>")
else:
  sequence_warning = widgets.HTML(" ")

sequence_input = widgets.Textarea(
    placeholder="protein sequence like 'VSVELPAPSSWKKLFYPNKVGSVKKTEVVFVAPTGEEISNRKQLEQYLKSHPGNPAIAEFDWTTSG'",
    description="Sequence:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%", height="100px"),
    value=prefill_seq,
)

use_filtering = widgets.Checkbox(value=True, description="Use filtering")
use_env_seqs = widgets.Checkbox(value=False, description="Use environmental sequences")

msa_run_button = widgets.Button(
    description="Load MSA",
    button_style="primary",
    icon="check",
    style=widgets.ButtonStyle(button_color="#1a73e8")
)

msa_cell_output = widgets.Output()

msa_upload_section = widgets.VBox([msa_file_upload])
msa_uniprot_section = widgets.VBox([msa_uniprot_id_input])
mmseqs_section = widgets.VBox([
    sequence_input,
    sequence_warning,
    widgets.HTML("<b>MMSeqs2 options</b>"),
    use_filtering,
    use_env_seqs,
])


def on_msa_method_change(change):
    method = change["new"]
    msa_upload_section.layout.display = "flex" if method == "upload_msa_local_file" else "none"
    msa_uniprot_section.layout.display = "flex" if method == "fetch_msa_from_alphafold_db" else "none"
    mmseqs_section.layout.display = "flex" if method == "run_mmseqs2_api" else "none"

load_msa_method_dropdown.observe(on_msa_method_change, names="value")
on_msa_method_change({"new": load_msa_method_dropdown.value})

def on_msa_run_button_clicked(b):
    with msa_cell_output:
        clear_output()

        global msa_path, target_sequence

        upload_method = load_msa_method_dropdown.value
        msa_path = None

        if upload_method == "upload_msa_local_file":
            if not msa_file_upload.value:
                print("❌ Please upload a file first.")
                return
            uploaded_file = list(msa_file_upload.value.values())[0]
            msa_path = uploaded_file["metadata"]["name"]
            if not any(msa_path.endswith(ext) for ext in FastaStream.ACCEPTED_MSA_EXTENSIONS):
                print(f"❌ Uploaded MSA file '{msa_path}' should have extension among {FastaStream.ACCEPTED_MSA_EXTENSIONS}.")
                msa_path = None
                return
            with open(msa_path, "wb") as f:
                f.write(uploaded_file["content"])
            print(f" * ✅ MSA file uploaded: '{msa_path}'")

        elif upload_method == "fetch_msa_from_alphafold_db":
            uniprot_id = msa_uniprot_id_input.value.strip()
            if not uniprot_id:
                print("❌ Please specify a UniProt ID.")
                return
            msa_path = fetch_msa_by_uniprot(uniprot_id)
            print(f" * ✅ MSA file fetched from AF-DB: '{msa_path}'")

        elif upload_method == "run_mmseqs2_api":
            query_sequence = sequence_input.value.strip().replace(" ", "").upper()
            if not query_sequence:
                print("❌ Please specify a protein sequence.")
                return
            if not all(aa in "ACDEFGHIKLMNPQRSTVWY" for aa in query_sequence):
                print("❌ Sequence contains invalid amino acids.")
                return

            # set query name to PDB name if it exists
            if "pdb_name" in globals():
              query_name = pdb_name
            else:
              query_name = "query"

            api = MMSeqs2API(
                f"./{query_name}_mmseqs2_out",
                use_env=use_env_seqs.value,
                use_filter=use_filtering.value,
            )
            msa_path = api.run(query_sequence, f"./{query_name}.a3m", query_name)
            print(f" * ✅ MSA file obtained from MMSeqs2 API: '{msa_path}'")

        target_sequence = FastaReader.read_first_sequence(msa_path)
        print(f" * ✅ Target sequence (L={len(target_sequence)}): '{target_sequence.sequence}'")

msa_run_button.on_click(on_msa_run_button_clicked)

display(widgets.VBox([
    widgets.HTML("<h2>MSA options</h2>"),
    load_msa_method_dropdown,
    msa_upload_section,
    msa_uniprot_section,
    mmseqs_section,
    msa_run_button,
    msa_cell_output,
]))

In [ ]:
#@title Select PDB chain
chain = "A" # @param {"type":"string","placeholder":"select a single chains like 'A'"}

# Input guardians
if msa_path == "" or msa_path is None:
  raise ValueError(f"❌ ERROR: msa_path is not set: Please first select an MSA.")
if pdb_path == "" or pdb_path is None:
  raise ValueError(f"❌ ERROR: pdb_path is not set: Please first, select a 3D structure.")

# Guardians
if len(chain) != 1:
  chain_failed = chain
  chain = None
  raise ValueError(f"❌ chain='{chain_failed}' should be a string of length 1.")
if chain not in sequences_by_chain:
  chain_failed = chain
  chain = None
  raise ValueError(f"❌ chain='{chain_failed}' not in PDB '{pdb_path}' (among chains '{''.join(sequences_by_chain.keys())}')")

# Show alignments
print(f" * ✅ MSA target sequence aligned to chain '{chain}' in PDB structure.")
align = PairwiseAlignment(target_sequence, sequences_by_chain[chain])
align.show();

# Not perfect alignments warning
if align.match_ratio == 1.0:
  print(f" * ✅ Perfect alignment between MSA-sequence vs. 3D-structure.")
else:
  all_chains = "".join(list(sequences_by_chain.keys()))
  msg = (
      f"\n⚠️ WARNING: MSA-sequence vs. 3D-structure alignment is not perfect: \n"
      f"    - 🟢 match:         {align.match} \n"
      f"    - 🟡 tail gaps:     {align.tail_gap} \n"
      f"    - 🟠 internal gaps: {align.internal_gap} \n"
      f"    - 🔴 mismatch:      {align.mismatch} \n"
      f" -> Please verify if selected chain '{chain}' is correct (among chains '{all_chains}')"
  )
  print(msg)


In [ ]:
#@title Run StructureDCA
import psutil

# StructureDCA run settings
#@markdown ### StructureDCA settings
#@markdown Filter low pLDDT regions from contacts? (enable if AlphaFold structure)
use_contacts_plddt_filter = False # @param {"type": "boolean"}
#@markdown Distance threshold (Å) for the contact map:
distance_cutoff = 8.00 # @param {"type":"number","placeholder":"8.00"}
#@markdown Remove MSA sequences whose seq. id. with target is lower than...:
min_seqid = 0.25 # @param {"type":"number","placeholder":"0.25"}
#@markdown Reweight MSA sequences sharing seq. id. higher than...:
weights_seqid = 0.8 # @param {"type":"number","placeholder":"0.8"}
#@markdown Exclude gaps from the DCA model? (if not, gap symbol is considered as the 21st amino acid)
exclude_gaps = True # @param {"type": "boolean"}
#@markdown Keep target sequence in the DCA model?
count_target_sequence = True # @param {"type": "boolean"}


# Input guardians
if msa_path == "" or msa_path is None:
  raise ValueError(f"❌ ERROR: msa_path is not set: Please first select an MSA.")
if pdb_path == "" or pdb_path is None:
  raise ValueError(f"❌ ERROR: pdb_path is not set: Please first, select a 3D structure.")
if chain == "" or chain is None:
  raise ValueError(f"❌ ERROR: chain is not set: Please first, select a chain.")


sdca = StructureDCA(
    msa_path=msa_path,
    pdb_path=pdb_path,
    chains=chain,
    distance_cutoff=distance_cutoff,
    use_contacts_plddt_filter=use_contacts_plddt_filter,
    exclude_gaps=exclude_gaps,
    min_seqid=min_seqid,
    weights_seqid=weights_seqid,
    count_target_sequence=count_target_sequence,
    num_threads=max(1, psutil.cpu_count())
)

sdca_scores = sdca.eval_mutations_table(
    log_output_sample=False,
)

In [ ]:
# Download predictions
from typing import List, Dict, Union
from littlecsv import CSV

def save_mutations_table_to_csv(scores: List[Dict[str, Union[str, float]]], save_path: str, sep: str) -> None:
    scores_csv = CSV(list(scores[0].keys()), sep=sep)
    for score_entry in scores:
        score_entry_copy = {k: v for k, v in score_entry.items()}
        for prop in ["RSA", "pLDDT", "gap_ratio"]: # post-process properties that may be lists before saving to .csv file
            score_entry_copy[prop] = ":".join([f"{x:.2f}" for x in score_entry_copy[prop]])
        scores_csv.add_entry(score_entry_copy)
        scores_csv.write(save_path)


#@markdown ### ▶️ Download all single-site predictions as csv
output_filename = "" # @param {"type":"string","placeholder":"<pdb_name>_sdca.csv"}
#@markdown Separator in output CSV file:
sep = "," # @param {"type":"string","placeholder":"CSV separator like ',' or ';'"}

# Input guardians --------------------------------------------------------------
if "sdca_scores" not in globals():
  raise RuntimeError(f"❌ ERROR: Please run StructureDCA first.")

if output_filename == "" or output_filename is None:
  output_filename = f"{pdb_name}_sdca.csv"

# Save predictions
output_path = output_filename
save_mutations_table_to_csv(sdca_scores, output_path, sep)
files.download(output_filename)

In [ ]:
#@title Heatmap of StructureDCA mutational predictions

# Imports ----------------------------------------------------------------------
import shutil
import io
import zipfile
import base64
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from IPython.display import display, HTML

# Input guardians --------------------------------------------------------------
if "sdca_scores" not in globals():
  raise RuntimeError(f"❌ ERROR: Please run StructureDCA first.")


# Plot Settings ----------------------------------------------------------------

# Sizes settings
CELL_SIZE = 60
DPI = 100
MIN_SIZE = 25
RANGE_SIZE = 100
TICKS_FONTSIZE = 20
VAL_FONTSIZE = 12
NOTE_FONTSIZE = 22

# Colors
color_fill = (0.90, 0.90, 0.90)

cmap_rsa = mcolors.LinearSegmentedColormap.from_list(
    "RSA",
    [
        (0.0, 0.373, 0.451),
        (1.0, 1.0, 1.0),
    ],
)
cmap_rsa.set_bad(color=color_fill)

cmap_gap_ratio = mcolors.LinearSegmentedColormap.from_list(
    "gap_ratio",
    [
        (0.000, 0.325, 0.839),
        (0.396, 0.796, 0.953),
        (1.000, 0.859, 0.075),
        (1.000, 0.490, 0.271),
        (1.000, 0.271, 0.271),
    ],
)
cmap_gap_ratio.set_bad(color=color_fill)

cmap_plddt = mcolors.ListedColormap([
    (1.000, 0.490, 0.271),
    (1.000, 0.859, 0.075),
    (0.396, 0.796, 0.953),
    (0.000, 0.325, 0.839)
])
cmap_plddt.set_bad(color=color_fill)
norm_plddt = mcolors.BoundaryNorm([0, 50, 70, 90, 100], cmap_plddt.N)

cmap = plt.get_cmap('RdBu').reversed()
cmap.set_bad(color=color_fill)


# Extract StructureDCA values --------------------------------------------------

aas = [aa.one for aa in AminoAcid.get_all()]
A = len(aas)
L = sdca.msa_length
Ntot = sdca.msa_depth
sequence = sdca.target_sequence.sequence

print("Plot Heatmap of StructureDCA single mutations predictions")

scores_map = {m["mutation_fasta"]: m for m in sdca_scores}
plddt = np.full(L, np.nan, dtype=np.float32)
rsa = np.full(L, np.nan, dtype=np.float32)
gap_perc = np.full(L, np.nan, dtype=np.float32)
sdca_vector = np.full(L, np.nan, dtype=np.float32)
rsadca_vector = np.full(L, np.nan, dtype=np.float32)
sdca_matrix = np.full((A, L), np.nan, dtype=np.float32)
rsasdca_matrix = np.full((A, L), np.nan, dtype=np.float32)
for i, wt_aa in enumerate(sequence):
  sdca_arr: list[float] = []
  rsasdca_arr: list[float] = []
  for j, mt_aa in enumerate(aas):
    mut = f"{wt_aa}{str(i+1)}{mt_aa}"
    mut_scores = scores_map[mut]
    plddt[i] = mut_scores["pLDDT"][0]
    rsa[i] = mut_scores["RSA"][0]
    gap_perc[i] = mut_scores["gap_ratio"][0] * 100.0
    sdca_score = mut_scores["StructureDCA"]
    rsasdca_score = mut_scores["RSA*StructureDCA"]
    sdca_matrix[j, i] = sdca_score
    rsasdca_matrix[j, i] = rsasdca_score
    if sdca_score is not None:
      sdca_arr.append(sdca_score)
    if rsasdca_score is not None:
      rsasdca_arr.append(rsasdca_score)
  if len(sdca_arr) > 0:
    sdca_vector[i] = np.mean(sdca_arr)
  if len(rsasdca_arr) > 0:
    rsadca_vector[i] = np.mean(rsasdca_arr)
sdca_extreme_val = np.nanmax(np.abs(sdca_matrix))
rsadca_extreme_val = np.nanmax(np.abs(rsasdca_matrix))


# Plot Heatmap -----------------------------------------------------------------

predictions_data = [
    ("StructureDCA", sdca_matrix, sdca_vector, sdca_extreme_val),
    ("RSA*StructureDCA", rsasdca_matrix, rsadca_vector, rsadca_extreme_val),
]

# Collect PNG bytes for all figures
png_files: dict[str, bytes] = {}  # filename -> bytes

for prediction, matrix, mean_vector, extreme_val in predictions_data:

  print("\n-----------------------------------------------------------------------")
  print(f"> {prediction}: ")

  segments: list[tuple[int, int]] = []
  segment_start = 0
  while segment_start + RANGE_SIZE < L:
      segments.append((segment_start, segment_start + RANGE_SIZE))
      segment_start += RANGE_SIZE
  segments.append((segment_start, L))

  for n_seg, segment in enumerate(segments):

    start, end = segment
    sequence_seg = sequence[start:end]
    L_seg = len(sequence_seg)
    gap_perc_seg = gap_perc[start:end]
    plddt_seg = plddt[start:end]
    rsa_seg = rsa[start:end]
    vector_seg = mean_vector[start:end]
    matrix_seg = matrix[:, start:end]
    SEGMENT_SIZE = RANGE_SIZE
    if len(segments) == 1:
      SEGMENT_SIZE = max(L_seg, MIN_SIZE)
    print(f"\n    - Range [{n_seg+1}/{len(segments)}]: {start+1} - {end}")

    fig_width = (SEGMENT_SIZE+9) * CELL_SIZE / DPI
    fig_height = (A+4) * CELL_SIZE / DPI
    fig = plt.figure(figsize=(fig_width*1.01, fig_height*1.10), dpi=DPI)
    gs = gridspec.GridSpec(nrows=5, ncols=1, height_ratios=[1, 1, 1, 1, A])
    ax_gap = fig.add_subplot(gs[0])
    ax_plddt = fig.add_subplot(gs[1])
    ax_rsa = fig.add_subplot(gs[2])
    ax_avg = fig.add_subplot(gs[3])
    ax0 = fig.add_subplot(gs[4])

    ax_gap.set_xticks([])
    ax_gap.set_yticks([0])
    ax_gap.set_yticklabels(["gap %"], fontsize=TICKS_FONTSIZE)
    gap_perc_seg_ = np.full(SEGMENT_SIZE, np.nan)
    gap_perc_seg_[:L_seg] = gap_perc_seg
    im_gap = ax_gap.imshow(np.array([gap_perc_seg_]), cmap=cmap_gap_ratio, vmin=0, vmax=100, aspect='equal')
    for i, i_gap in enumerate(gap_perc_seg):
      ax_gap.text(i, 0, str(int(i_gap)), ha='center', va='center', fontsize=VAL_FONTSIZE)

    ax_plddt.set_xticks([])
    ax_plddt.set_yticks([0])
    ax_plddt.set_yticklabels(["pLDDT"], fontsize=TICKS_FONTSIZE)
    plddt_seg_ = np.full(SEGMENT_SIZE, np.nan)
    plddt_seg_[:L_seg] = plddt_seg
    im_plddt = ax_plddt.imshow(np.array([plddt_seg_]), cmap=cmap_plddt, norm=norm_plddt, aspect='equal')
    for i, i_plddt in enumerate(plddt_seg):
      if np.isnan(i_plddt): continue
      ax_plddt.text(i, 0, str(int(i_plddt)), ha='center', va='center', fontsize=VAL_FONTSIZE)

    ax_rsa.set_xticks([])
    ax_rsa.set_yticks([0])
    ax_rsa.set_yticklabels(["RSA"], fontsize=TICKS_FONTSIZE)
    rsa_seg_ = np.full(SEGMENT_SIZE, np.nan)
    rsa_seg_[:L_seg] = rsa_seg
    im_rsa = ax_rsa.imshow(np.array([rsa_seg_]), cmap=cmap_rsa, vmin=0, vmax=100, aspect='equal')
    for i, i_rsa in enumerate(rsa_seg):
      if np.isnan(i_rsa): continue
      ax_rsa.text(i, 0, str(int(i_rsa)), ha='center', va='center', fontsize=VAL_FONTSIZE)

    ax_avg.set_xticks([])
    ax_avg.set_yticks([0])
    ax_avg.set_yticklabels(["Mean"], fontsize=TICKS_FONTSIZE)
    vector_seg_ = np.full(SEGMENT_SIZE, np.nan)
    vector_seg_[:L_seg] = vector_seg
    im_avg = ax_avg.imshow(np.array([vector_seg_]), cmap=cmap, vmin=-extreme_val, vmax=extreme_val, aspect='equal')
    for i, i_avg in enumerate(vector_seg):
      if np.isnan(i_avg): continue
      ax_avg.text(i, 0, f"{i_avg:.1f}", ha='center', va='center', fontsize=VAL_FONTSIZE)

    matrix_img = np.full((A, SEGMENT_SIZE), np.nan, dtype=np.float32)
    for i, wt_aa in enumerate(sequence_seg):
      for j, mt_aa in enumerate(aas):
        if wt_aa == mt_aa:
          x = "-"
          matrix_img[j, i] = np.nan
        else:
          matrix_img[j, i] = matrix_seg[j, i]
          x_float = matrix_seg[j, i]
          if np.isnan(x_float): continue
          x = f"{x_float:.1f}"
        ax0.text(i, j, x, ha='center', va='center', fontsize=VAL_FONTSIZE)
    im0 = ax0.imshow(matrix_img, cmap=cmap, vmin=-extreme_val, vmax=extreme_val, aspect='equal')
    ax0.set_xticks([i for i in range(len(sequence_seg))])
    ax0.set_xticklabels([f"{sequence_seg[i]} {str(start+i+1)}" for i in range(len(sequence_seg))], rotation=90, fontsize=TICKS_FONTSIZE)
    ax0.set_yticks([i for i in range(A)])
    ax0.set_yticklabels([aa for aa in aas], fontsize=TICKS_FONTSIZE)

    plt.tight_layout()
    plt.show()

    # Save figure to bytes
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=DPI, bbox_inches="tight")
    buf.seek(0)
    fname = f"{pdb_name}_{prediction}_range{start+1}-{end}.png"
    png_files[fname] = buf.read()
    plt.close(fig)


# Download button --------------------------------------------------------------
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, "w", zipfile.ZIP_DEFLATED) as zf:
  for fname, png_bytes in png_files.items():
    zf.writestr(fname, png_bytes)
zip_buf.seek(0)
b64 = base64.b64encode(zip_buf.read()).decode()
zip_filename = f"{pdb_name}_heatmaps.zip"
display(HTML(f"""
    <a download="{zip_filename}" href="data:application/zip;base64,{b64}">
        <button style="margin-top:10px; padding:8px 16px; background:#1a73e8; color:white; border:none; border-radius:4px; cursor:pointer; font-size:14px;">
            Download heatmaps ({len(png_files)} PNG files)
        </button>
    </a>
"""))

In [ ]:
#@title 3D Viewer


# User input -------------------------------------------------------------------
#@markdown ### Color residues by:
metric = "StructureDCA" # @param ["StructureDCA", "RSA*StructureDCA", "RSA", "pLDDT", "gap_freq"]
#@markdown ---------------------------------------------------------------------
#@markdown ### Display settings
show_sticks = True # @param {"type":"boolean"}
sticks_radius = 0.15 # @param {"type":"number","placeholder":"a positive number like: 0.15"}

# Input guardians --------------------------------------------------------------
if "sdca_scores" not in globals():
  raise RuntimeError(f"❌ ERROR: Please run StructureDCA first.")


# Install and Imports ----------------------------------------------------------
try:
    import py3Dmol
except ImportError:
    %pip install py3Dmol
    import py3Dmol
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import base64
from IPython.display import display, HTML


# Extract StructureDCA values --------------------------------------------------------

# Log
all_chains = "".join(list(sequences_by_chain.keys()))
print(f"Display {metric} for 3D Structure: '{clip_string(pdb_name)}' ✅")
print(f" * target chain: '{chain}'")
print(f" * all chains:   '{all_chains}'")

# Extract maps
rsadca_map: dict[str, float] = {}
sdca_map: dict[str, float] = {}
rsa_map: dict[str, float] = {}
plddt_map: dict[str, float] = {}
gap_freq_map: dict[str, float] = {}
for mutation_entry in sdca_scores:
  mut = mutation_entry["mutation_pdb"]
  if mut == "XXX" or mut == "" or mut is None:
    continue
  resid = mut[1:-1]
  rsasdca_score = mutation_entry["RSA*StructureDCA"]
  sdca_score = mutation_entry["StructureDCA"]
  rsa = mutation_entry["RSA"][0]
  plddt = mutation_entry["pLDDT"][0]
  gap_freq = mutation_entry["gap_ratio"][0]
  if rsasdca_score is not None:
    if resid not in rsadca_map:
      rsadca_map[resid] = []
    rsadca_map[resid].append(rsasdca_score)
  if sdca_score is not None:
    if resid not in sdca_map:
      sdca_map[resid] = []
    sdca_map[resid].append(sdca_score)
  if rsa is not None:
    rsa_map[resid] = rsa
  if plddt is not None:
    plddt_map[resid] = plddt
  if gap_freq is not None:
    gap_freq_map[resid] = gap_freq
rsadca_map = {k: np.mean(v) for k, v in rsadca_map.items()}
sdca_map = {k: np.mean(v) for k, v in sdca_map.items()}
rsadca_extreme_val = np.max(np.abs(list(rsadca_map.values())))
sdca_extreme_val = np.max(np.abs(list(sdca_map.values())))


# Colors Settings --------------------------------------------------------------
color_other_chains = (0.70, 0.70, 0.70)
color_target_chain = (0.50, 0.50, 0.50)

cmap_rsalor = plt.get_cmap('RdBu').reversed()
cmap_rsalor.set_bad(color=color_target_chain)
norm_rsalor = mcolors.Normalize(vmin=-rsadca_extreme_val, vmax=rsadca_extreme_val, clip=True)

cmap_lor = plt.get_cmap('RdBu').reversed()
cmap_lor.set_bad(color=color_target_chain)
norm_lor = mcolors.Normalize(vmin=-sdca_extreme_val, vmax=sdca_extreme_val, clip=True)

cmap_rsa = mcolors.LinearSegmentedColormap.from_list(
    "RSA",
    [
        (0.0, 0.373, 0.451),
        (1.0, 1.0, 1.0),
    ],
)
cmap_rsa.set_bad(color=color_target_chain)
norm_rsa = mcolors.Normalize(vmin=0.0, vmax=100.0, clip=True)

cmap_gap_freq = mcolors.LinearSegmentedColormap.from_list(
    "gap_freq",
    [
        (0.000, 0.325, 0.839),
        (0.396, 0.796, 0.953),
        (1.000, 0.859, 0.075),
        (1.000, 0.490, 0.271),
        (1.000, 0.271, 0.271),
    ],
)
cmap_gap_freq.set_bad(color=color_target_chain)
norm_gap_freq = mcolors.Normalize(vmin=0.0, vmax=1.0)

cmap_plddt = mcolors.ListedColormap([
    (1.000, 0.490, 0.271),
    (1.000, 0.859, 0.075),
    (0.396, 0.796, 0.953),
    (0.000, 0.325, 0.839)
])
cmap_plddt.set_bad(color=color_target_chain)
norm_plddt = mcolors.BoundaryNorm([0, 50, 70, 90, 100], cmap_plddt.N)


# 3D viewer --------------------------------------------------------------------

values_map = {
  "StructureDCA": (
      sdca_map, cmap_lor, norm_lor,
      "0 is neutral, higher values are deleterious / destabilizing",
    ),
  "RSA*StructureDCA": (
      rsadca_map, cmap_rsalor, norm_rsalor,
      "0 is neutral, higher values are deleterious / destabilizing",
    ),
  "RSA": (
      rsa_map, cmap_rsa, norm_rsa,
      "0 is fully burried, 100 is fully exposed",
    ),
  "pLDDT": (
      plddt_map, cmap_plddt, norm_plddt,
      "higher values are better \n(if structure is AlphaFold or similar model, otherwise it is B-factor)",
    ),
  "gap_freq": (
      gap_freq_map, cmap_gap_freq, norm_gap_freq,
      "0 means no gaps, 1 means all gaps",
    ),
}
residue_values, cmap, norm, scale_comment = values_map[metric]

residue_colors = {
  resid: mcolors.to_hex(cmap(norm(v)))
  for resid, v in residue_values.items()
}

# Plot color scale
vmin, vmid, vmax = norm.vmin, (norm.vmin + norm.vmax) / 2.0, norm.vmax
gradient = np.linspace(vmin, vmax, 100+1).reshape(1, -1)
fig, ax = plt.subplots(figsize=(5, 0.25))
im = ax.imshow(gradient, aspect="auto", cmap=cmap, norm=norm)
ax.set_yticks([])
ax.set_xticks([0, 50, 100], [f"{vmin:.2f}", f"{vmid:.2f}" ,f"{vmax:.2f}"])
ax.set_title(scale_comment, fontsize=10)
plt.show()
print("")

# 3D viewer
with open(pdb_path, "r") as fs:
  pdb_data = fs.read()
view = py3Dmol.view(data=pdb_data)
view.setStyle(
  {'cartoon': {'color': mcolors.to_hex(color_other_chains)}}
)
if show_sticks:
  view.addStyle(
    {'stick': {'color': mcolors.to_hex(color_other_chains), 'radius': sticks_radius}}
  )
view.setStyle(
  {'chain': chain},
  {'cartoon': {'color': mcolors.to_hex(color_target_chain)}}
)
if show_sticks:
  view.addStyle(
    {'chain': chain},
    {'stick': {'color': mcolors.to_hex(color_target_chain), 'radius': sticks_radius}}
  )
for resid, color in residue_colors.items():
  chain, res_position = resid[0], resid[1:]
  view.setStyle(
    {'chain': chain, 'resi': res_position},
    {'cartoon': {'color': color}}
  )
  if show_sticks:
    view.addStyle(
      {'chain': chain, 'resi': res_position},
      {'stick': {'color': color, 'radius': sticks_radius}}
    )
view.zoomTo()
view.show()

# Download button --------------------------------------------------------------
html_content = view._make_html()
filename = f"{pdb_name}_{metric}_3Dviewer.html"
b64 = base64.b64encode(html_content.encode()).decode()
download_button = HTML(f"""
    <a download="{filename}" href="data:text/html;base64,{b64}">
        <button style="margin-top:10px; padding:8px 16px; background:#1a73e8; color:white; border:none; border-radius:4px; cursor:pointer; font-size:14px;">
            Download as HTML
        </button>
    </a>
""")
display(download_button)

In [ ]:
#@markdown ### Multiple-site prediction

default_output_name = "predictions.csv"

mut_load_method_dropdown = widgets.Dropdown(
    options=[
        ("Enter mutations", "enter_mutations"),
        ("Upload", "upload_mutations_file"),
    ],
    value="enter_mutations",
    description="Method:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

mut_enter_input = widgets.Textarea(
    placeholder="Enter one mutation per line",
    description="Mutations:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px", height="150px")
)

mut_file_upload = widgets.FileUpload(
    multiple=False,
    description="Upload mutations file...",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

mut_run_button = widgets.Button(
    description="Run",
    button_style="primary",
    icon="check",
    style=widgets.ButtonStyle(button_color="#1a73e8")
)

mut_file_download = widgets.Text(
    value=default_output_name,
    description="File output name:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

mut_cell_output = widgets.Output()

# msa pos
mut_enter_section = widgets.VBox([
    widgets.HTML("Please provide as mutation strings in FASTA convention, one mutation per line"),
    mut_enter_input
    ])

mut_upload_section = widgets.VBox([
    widgets.HTML("Please upload a file with mutation strings in FASTA convention, one mutation per line"),
    mut_file_upload
    ])


def on_mut_method_change(change):
    method = change["new"]
    mut_upload_section.layout.display = "flex" if method == "upload_mutations_file" else "none"
    mut_enter_section.layout.display = "flex" if method == "enter_mutations" else "none"

mut_load_method_dropdown.observe(on_mut_method_change, names="value")
on_mut_method_change({"new": mut_load_method_dropdown.value})


def on_mut_run_button_clicked(b):
    with mut_cell_output:
        clear_output()

        upload_method = mut_load_method_dropdown.value

        if upload_method == "upload_mutations_file":
          if not mut_file_upload.value:
            print("❌ Please upload a file first.")
            return
          uploaded_file = list(mut_file_upload.value.values())[0]
          input_file_path = uploaded_file['metadata']['name']
          with open(input_file_path, "wb") as f:
                f.write(uploaded_file["content"])
          mutations = read_mutations_file(input_file_path)

        elif upload_method == "enter_mutations":
            try:
              mutations = [m.strip() for m in mut_enter_input.value.splitlines() if m.strip()]
            except:
                print("❌ Please enter mutations")
                return

        output_filename = mut_file_download.value
        if output_filename == "":
          output_filename = default_output_name


        sdca.eval_mutations_table(
            mutations,
            save_path=output_filename
            )
        files.download(output_filename)

mut_run_button.on_click(on_mut_run_button_clicked)

display(widgets.VBox([
    mut_load_method_dropdown,
    mut_enter_section,
    mut_upload_section,
    mut_file_download,
    mut_run_button,
    mut_cell_output,
]))

if "sdca" not in globals():
  raise NameError(f"❌ ERROR: Please run StructureDCA first.")

In [ ]:
#@markdown ### Evaluate a sequence
seq_to_evaluate = "" # @param {"type":"string","placeholder":"Enter a sequence to evaluate"}

if "sdca" not in globals():
  raise RuntimeError(f"❌ ERROR: Please run StructureDCA first.")

if len(seq_to_evaluate)==0:
  raise ValueError(f"❌ ERROR: Please enter a sequence to evaluate.")

sdca_score = sdca.eval_sequence(seq_to_evaluate, reweight_by_rsa=False)
rsasdca_score = sdca.eval_sequence(seq_to_evaluate, reweight_by_rsa=True)

display(f"StructureDCA     : {sdca_score:.2f}")
display(f"RSA*StructureDCA : {rsasdca_score:.2f}")